In [1]:
import os
import re
from typing import List, Literal, Tuple
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from IPython.display import display, HTML

load_dotenv()

FeatureState = Literal["Yes", "No", "Blank"]

# =====================================================================
# 1. Pydantic Output Schema
# =====================================================================
class FeatureAuditResult(BaseModel):
    f1_individual_questions: FeatureState = Field(
        description="Yes: Tutor asked questions calling at least two different students individually by name. No: Generic group questions only or only 1 dominant student. Blank: Neutral."
    )
    f1_evidence: str

    f2_homework_given: FeatureState = Field(
        description="Yes: Explicit practice tasks/puzzles assigned before class ended. No: Class ended without homework. Blank: Neutral."
    )
    f2_evidence: str

    f_rapport_building: FeatureState = Field(
        description="Yes: Warm non-chess personal banter/check-ins. No: Jumps straight into revision/moves without building rapport. Blank: Neutral greeting."
    )
    f_rapport_evidence: str

    f3_solid_intro: FeatureState = Field(
        description="Yes: Solid introduction / clear orientation of lesson goals. No: Lacked good intro. Blank: Neutral."
    )
    f3_evidence: str

    f4_setup_and_puzzles: FeatureState = Field(
        description="Yes: Actively coached learners during setup/puzzles. No: Passive silence during exercises. Blank: No setup/puzzle phase."
    )
    f4_evidence: str

    f5_interactive_flow: FeatureState = Field(
        description="Yes: Steady 2-way dialogue throughout. No: Monologues or dead air. Blank: Neutral."
    )
    f5_evidence: str

    f6_conclusion_quality: FeatureState = Field(
        description="No: Abrupt end without closing wrap-up. Yes: Clear wrap-up provided. Blank: Standard end."
    )
    f6_evidence: str

    f7_joiner_intros: FeatureState = Field(
        description="Yes: Dedicated time for new student to introduce themselves. No: New student present/trial but skipped. Blank: No new joiners."
    )
    f7_evidence: str

    students_called: List[str]


# =====================================================================
# 2. Extraction Prompt & Pure-Python Parser
# =====================================================================
EXTRACTION_SYSTEM_PROMPT = """You are an expert pedagogical auditor evaluating online children's chess classes.
Audit the transcript dialogue and return a single valid JSON object strictly matching this format:

{
  "f1_individual_questions": "Yes" | "No" | "Blank",
  "f1_evidence": "direct quote or observation",
  "f2_homework_given": "Yes" | "No" | "Blank",
  "f2_evidence": "direct quote or observation",
  "f_rapport_building": "Yes" | "No" | "Blank",
  "f_rapport_evidence": "direct quote or observation",
  "f3_solid_intro": "Yes" | "No" | "Blank",
  "f3_evidence": "direct quote or observation",
  "f4_setup_and_puzzles": "Yes" | "No" | "Blank",
  "f4_evidence": "direct quote or observation",
  "f5_interactive_flow": "Yes" | "No" | "Blank",
  "f5_evidence": "direct quote or observation",
  "f6_conclusion_quality": "Yes" | "No" | "Blank",
  "f6_evidence": "direct quote or observation",
  "f7_joiner_intros": "Yes" | "No" | "Blank",
  "f7_evidence": "direct quote or observation",
  "students_called": ["Name1", "Name2"]
}

Strict Rules:
- f1_individual_questions: 'Yes' ONLY if tutor calls at least two different students by name to answer. If whole room only, mark 'No'.
- f2_homework_given: 'Yes' ONLY if explicit practice tasks or puzzles are assigned before ending. If class ends without tasks, mark 'No'.
- f_rapport_building: 'Yes' ONLY if authentic social check-in took place (school, weekend, hobbies). If tutor jumped straight into chess, mark 'No'.
- f7_joiner_intros: 'Yes' if new student was invited to introduce themselves. 'No' if new joiner was ignored. 'Blank' if no new joiners.
- Return ONLY the JSON object. Do not include introductory text, explanations, or trailing commentary.
"""

def get_groq_llm():
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        raise ValueError("GROQ_API_KEY environment variable is not set.")
    
    return ChatGroq(
        model_name="qwen/qwen3.8-27b",  # or "openai/gpt-oss-120b"
        groq_api_key=api_key,
        temperature=0.0
    )

def parse_llm_json_response(raw_response) -> FeatureAuditResult:
    """
    Extracts text from content, content-blocks, or reasoning buffers,
    then parses the JSON object.
    """
    # 1. Extract raw text from whichever field Groq populated
    if hasattr(raw_response, "content"):
        content = raw_response.content
        if isinstance(content, list):
            content = "".join([c.get("text", "") if isinstance(c, dict) else str(c) for c in content])
        if not content.strip():
            # Check for thinking/reasoning model buffers
            add_kw = getattr(raw_response, "additional_kwargs", {})
            content = add_kw.get("reasoning_content", "") or add_kw.get("reasoning", "")
    else:
        content = str(raw_response)

    # 2. Extract JSON substring
    match = re.search(r"\{.*\}", content, re.DOTALL)
    if not match:
        raise ValueError(f"Could not locate JSON block in response (len={len(content)}): {content[:300]}")
    
    json_str = match.group(0)
    return FeatureAuditResult.model_validate_json(json_str)


# =====================================================================
# 3. Transcript Windowing (Token Limit Protection)
# =====================================================================
def extract_compact_payload(raw_vtt: str) -> str:
    lines = [
        l.strip() for l in raw_vtt.splitlines()
        if l.strip() and not l.startswith("WEBVTT") and not l.isdigit() and "-->" not in l
    ]
    full_text = "\n".join(lines)
    
    opening = full_text[:4000]       # First ~10 mins: Rapport, Intro, Joiners
    closing = full_text[-3500:]      # Last ~10 mins: Homework, Conclusion
    
    mid_lines = full_text[4000:-3500].splitlines()
    mid_sample = [
        l for l in mid_lines
        if any(w in l.lower() for w in ["?", "puzzle", "move", "can you", "what is", "try", "white to play"])
    ][:35]
    mid_sample_text = "\n".join(mid_sample)
    
    return f"""=== CLASS OPENING (MINUTES 0-10) ===
{opening}

=== MID-CLASS INTERACTIVE & PUZZLE EXCHANGES ===
{mid_sample_text}

=== CLASS CLOSING & HOMEWORK (FINAL 10 MINUTES) ===
{closing}"""


# =====================================================================
# 4. Anti-Bias Combinatorial Scoring Logic
# =====================================================================
def compute_score_and_rule(audit: FeatureAuditResult) -> Tuple[int, str, str]:
    f1 = audit.f1_individual_questions
    f2 = audit.f2_homework_given
    rap = audit.f_rapport_building
    f3 = audit.f3_solid_intro
    f4 = audit.f4_setup_and_puzzles
    f5 = audit.f5_interactive_flow
    f6 = audit.f6_conclusion_quality
    f7 = audit.f7_joiner_intros

    # STEP 1: Veto Gate - Negative Penalties Evaluated First
    if rap == "No" and f7 == "No":
        return 1, "Low Value", "Skipped rapport for revision + missed joiner intro"

    if rap == "No":
        return 1, "Low Value", "Directly entered class without building rapport"

    if f3 == "No" and f6 == "No":
        return 1, "Low Value", "Lacked good intro/conclusion and joiner introductions"

    # STEP 2: Strict High Value Criteria (Score 4)
    # Rule: Missing homework (f2 == "No") strictly prevents Score 4
    if f2 != "No":
        if f1 == "Yes" and f2 == "Yes":
            return 4, "High Value", "Questions asked individually by name + HW was given"
        if rap == "Yes" and f1 == "Yes":
            return 4, "High Value", "Good rapport building + asked questions individually"

    # STEP 3: Medium Value Coverage (Score 3)
    if f3 == "Yes" and f1 == "Yes" and f2 == "No":
        return 3, "Medium Value", "Good intro and included everyone, but missed HW"

    if f1 == "Yes" and f2 == "No":
        return 3, "Medium Value", "Maintained one-to-one queries, but should give HW"

    if f3 == "Yes" and f2 == "Yes":
        return 3, "Medium Value", "Included a solid introduction and assigned HW"

    if f4 == "Yes" and f2 == "Yes":
        return 3, "Medium Value", "Addressed learners during setup and puzzles; gave HW"

    if f5 == "Yes":
        return 3, "Medium Value", "General interactive flow was present"

    return 3, "Medium Value", "Standard baseline flow"


# =====================================================================
# 5. Pipeline Execution & Visualization
# =====================================================================
def audit_and_display(vtt_file_path: str):
    with open(vtt_file_path, "r", encoding="utf-8", errors="ignore") as f:
        compact_text = extract_compact_payload(f.read())

    llm = get_groq_llm()
    
    # Static SystemMessage prevents LangChain KeyError on literal JSON curly braces
    prompt = ChatPromptTemplate.from_messages([
        SystemMessage(content=EXTRACTION_SYSTEM_PROMPT),
        HumanMessagePromptTemplate.from_template("Class Transcript Dialogue:\n\n{transcript_text}")
    ])
    
    chain = prompt | llm
    raw_response = chain.invoke({"transcript_text": compact_text})
    
    extracted: FeatureAuditResult = parse_llm_json_response(raw_response.content)
    score, tier, rule = compute_score_and_rule(extracted)

    score_color = "#27ae60" if score == 4 else ("#e67e22" if score == 3 else "#c0392b")
    
    table_rows = [
        ("1", "Questions Asked Individually", extracted.f1_individual_questions, extracted.f1_evidence),
        ("2", "Homework Assignment", extracted.f2_homework_given, extracted.f2_evidence),
        ("3", "Rapport Building", extracted.f_rapport_building, extracted.f_rapport_evidence),
        ("4", "Introduction Quality", extracted.f3_solid_intro, extracted.f3_evidence),
        ("5", "Setup & Puzzles Engagement", extracted.f4_setup_and_puzzles, extracted.f4_evidence),
        ("6", "General Interactive Flow", extracted.f5_interactive_flow, extracted.f5_evidence),
        ("7", "Session Conclusion", extracted.f6_conclusion_quality, extracted.f6_evidence),
        ("8", "Joiner Introductions", extracted.f7_joiner_intros, extracted.f7_evidence),
    ]

    rows_html = ""
    for num, name, state, ev in table_rows:
        if state == "Yes":
            badge = '<span style="background: #e2efda; color: #276a3c; font-weight: bold; padding: 2px 8px; border-radius: 4px;">Yes</span>'
        elif state == "No":
            badge = '<span style="background: #fce4d6; color: #c65911; font-weight: bold; padding: 2px 8px; border-radius: 4px;">No</span>'
        else:
            badge = '<span style="color: #7f8c8d; padding: 2px 8px;">—</span>'
        
        rows_html += f"""
        <tr style="border-bottom: 1px solid #e5e7eb;">
            <td style="padding: 8px; text-align: center; font-weight: bold;">{num}</td>
            <td style="padding: 8px; font-weight: 500;">{name}</td>
            <td style="padding: 8px; text-align: center;">{badge}</td>
            <td style="padding: 8px; font-size: 13px; color: #374151;">{ev}</td>
        </tr>
        """

    display_html = f"""
    <div style="font-family: Arial, sans-serif; margin-bottom: 20px;">
        <div style="border-left: 5px solid {score_color}; padding: 10px 14px; background: #ffffff; box-shadow: 0 1px 3px rgba(0,0,0,0.1); border-radius: 4px; margin-bottom: 10px;">
            <h4 style="margin: 0; color: #111827;">File: {os.path.basename(vtt_file_path)}</h4>
            <p style="margin: 4px 0 0 0; font-size: 14px;">
                <b>Score:</b> <span style="color: {score_color}; font-weight: bold; font-size: 16px;">{score} / 4</span> ({tier}) &nbsp;|&nbsp; 
                <b>Rule:</b> <i>"{rule}"</i>
            </p>
            <p style="margin: 3px 0 0 0; font-size: 13px; color: #4b5563;">
                <b>Students Identified:</b> {", ".join(extracted.students_called) if extracted.students_called else "None"}
            </p>
        </div>
        <table style="width: 100%; border-collapse: collapse; border: 1px solid #e5e7eb; font-size: 13.5px;">
            <thead>
                <tr style="background: #1f4e79; color: white; text-align: left;">
                    <th style="padding: 8px; width: 40px; text-align: center;">#</th>
                    <th style="padding: 8px; width: 220px;">Feature</th>
                    <th style="padding: 8px; width: 80px; text-align: center;">State</th>
                    <th style="padding: 8px;">Evidence Quote</th>
                </tr>
            </thead>
            <tbody>
                {rows_html}
            </tbody>
        </table>
    </div>
    """
    try:
        display(HTML(display_html))
    except Exception:
        print(f"[{os.path.basename(vtt_file_path)}] Score: {score} ({tier}) | Rule: {rule}")
        
    return extracted, score, rule

# excellent

In [2]:
sample_vtt = r"class recordings_\excellent\B472 intermediate -transcript-1.1.vtt"

if os.path.exists(sample_vtt):
    extracted, score, rule = audit_and_display(sample_vtt)
else:
    print(f"File not found: {sample_vtt}")

#,Feature,State,Evidence Quote
1,Questions Asked Individually,No,"The tutor asks general questions to the room or specifically to Aagney (e.g., 'How was your day?'), but does not call on at least two different students by name to answer specific chess or academic questions. Pearson is greeted but not asked to answer a question in the same manner."
2,Homework Assignment,No,"The class ends with 'Okay, all the best, bye, good night.' No explicit practice tasks, puzzles, or homework are assigned before the tutor leaves."
3,Rapport Building,Yes,"Mayuri asks, 'So, how was your day?' and follows up with 'Was it good, bad? Best?' and discusses Aagney's piano classes and holidays."
4,Introduction Quality,No,The opening is disorganized with waiting for students to join and small talk. There is no structured introduction to the lesson topic or clear agenda setting before moving into casual chat and then directly to a puzzle link.
5,Setup & Puzzles Engagement,Yes,"Mayuri states, 'Okay, we'll have Puzzle Rush then. We'll not waste your time.' and later, 'Okay, I have, I have shared the, Puzzle Rish link.'"
6,General Interactive Flow,Yes,"The tutor engages in back-and-forth dialogue, checking on student positions in the tournament ('You're on the 4th position now'), reacting to game outcomes ('OMG', 'good luck for Ruthwik'), and maintaining a lively conversation throughout the puzzle segment."
7,Session Conclusion,No,"The class ends abruptly with 'Okay, all the best, bye, good night.' without summarizing the lesson, reviewing key points, or assigning follow-up tasks."
8,Joiner Introductions,Yes,"When Pearson joins, Mayuri says, 'Good evening, Pearson!' and later asks, 'Okay, that is, what is your name?' and 'So, what is your mother tongue?' inviting him to share personal details."


In [3]:
sample_vtt = r"class recordings_\excellent\B491 Beginner -transcript-1.1.vtt"

if os.path.exists(sample_vtt):
    extracted, score, rule = audit_and_display(sample_vtt)
else:
    print(f"File not found: {sample_vtt}")

#,Feature,State,Evidence Quote
1,Questions Asked Individually,Yes,"Tutor asks 'Awaan' to solve a puzzle ('Rock to D'), then asks 'Nisha' to try another ('Namaste, can you try this one?')."
2,Homework Assignment,No,The class ends with the tutor helping a student create an account and saying 'Bye next week' without assigning specific practice tasks or puzzles.
3,Rapport Building,Yes,"Tutor asks multiple students 'How was your day?' (e.g., to shilaf, Nisha, Avan) and acknowledges Gautham's story about checkmating his father."
4,Introduction Quality,Yes,"Tutor introduces the new student Midansh, asks for spelling and age, and asks the class to say hi to him."
5,Setup & Puzzles Engagement,Yes,Tutor sets up a 'puzzle race' via a link and later guides students through specific checkmate puzzles on the board.
6,General Interactive Flow,Yes,"Tutor actively manages the flow, switching between the online race and board puzzles, and prompting students for moves ('What about E8 Square?')."
7,Session Conclusion,No,"The ending is disorganized, focusing on technical account creation for one student while others leave, with no clear summary or homework assignment."
8,Joiner Introductions,Yes,Tutor invites Midansh to introduce himself ('Can you tell the spelling... 12 years old') and asks the class to greet him.


# GOOD

In [23]:
sample_vtt = r"class recordings_\good\B435 intermediate Part  -transcript-1.1.vtt"

if os.path.exists(sample_vtt):
    extracted, score, rule = audit_and_display(sample_vtt)
else:
    print(f"File not found: {sample_vtt}")

#,Feature,State,Evidence Quote
1,Questions Asked Individually,Yes,"Tutor calls multiple students by name to answer: 'Narain, what will… what is the best move here?', 'Ayaan, you're white.', 'Ahna, you're white?', 'HP: ...' (implied call or direct interaction), 'AAPTI, can you hear me?'"
2,Homework Assignment,No,"The tutor says 'You can continue doing the puzzles' and 'Link in the chat' but does not assign specific new practice tasks or puzzles as homework before ending the class; the class ends with 'We'll see you next week, bye!'"
—,Rapport Building,Yes,"Tutor asks 'Hi, how are you?' to Narain and HP, and asks HP 'Where were you?' when they were absent, indicating a social check-in."
3,Introduction Quality,Yes,"Tutor introduces the topic: 'So, today we are gonna learn a topic called the Sukswal, okay? Sukswal means, it is like a position towards the end of the game...'"
4,Setup & Puzzles Engagement,Yes,"Tutor sets up positions and asks for moves: 'Here, if we're white, and we try to go directly for the checkmate...', 'What will be the best move for black here?', 'Can you pause your videos and find a move for black...'"
5,General Interactive Flow,Yes,"Tutor engages students in real-time puzzle solving with countdowns and direct questions: '10. Nine. 8... 0...', 'Narain, what will… what is the best move here?', 'Ayaan, you're white.'"
6,Session Conclusion,No,"The conclusion is abrupt and lacks a clear summary or wrap-up of the lesson's key points; it ends with 'Okay, so you guys can continue, okay? We'll see you next week, bye!'"
7,Joiner Introductions,—,No new joiners are explicitly mentioned or invited to introduce themselves in the transcript.


In [24]:
sample_vtt = r"class recordings_\good\B552 Beginner -transcript-1.1.vtt"

if os.path.exists(sample_vtt):
    extracted, score, rule = audit_and_display(sample_vtt)
else:
    print(f"File not found: {sample_vtt}")

#,Feature,State,Evidence Quote
1,Questions Asked Individually,Yes,"Tutor asks 'Has Tataka done that?', 'How about Elden?', and 'Yes, Anika?'"
2,Homework Assignment,Yes,"Tutor says 'This will be your homework, okay? Just try a few minutes of this link.' and 'everyone should, practice this one, okay?'"
—,Rapport Building,No,"The tutor jumped straight into reviewing the previous lesson ('Yes, so what was the last class we took?') without asking about school, weekends, or hobbies."
3,Introduction Quality,Yes,"Tutor reviewed the previous lesson ('We learned, Special type of checkman') and introduced the new topic ('today, We'll be learning a new type of checkmate... Arabian meat')."
4,Setup & Puzzles Engagement,Yes,"Tutor shared the screen ('I'll be sharing the screen') and presented multiple puzzles ('So what will be the best move here?', 'we'll try a few of those puzzles as well')."
5,General Interactive Flow,Yes,"Students actively participated with answers like 'Yes, sir', 'Queen… No?', and 'Sir, the queen can go to H5 shop', and the tutor responded to each."
6,Session Conclusion,Yes,"Tutor summarized the lesson ('So, what did we learn today?'), assigned homework, previewed the next topic ('Borden's mate'), and said goodbye to students individually."
7,Joiner Introductions,No,"New students like Naveen Kumar joined and participated, but were not explicitly invited to introduce themselves by name at the start."


# AVERAGE


In [25]:
sample_vtt = r"class recordings_\average_\B435 Intermediate-transcript-1.1.vtt"

if os.path.exists(sample_vtt):
    extracted, score, rule = audit_and_display(sample_vtt)
else:
    print(f"File not found: {sample_vtt}")

#,Feature,State,Evidence Quote
1,Questions Asked Individually,Yes,"Tutor asks 'Eileen? Who will win in this position? What do you think?' and later 'She'll be a pro, and what about you, Anna?' and 'Now, what you will do, Narain?'"
2,Homework Assignment,No,"Tutor states 'Today, we will be, ending the session 5 minutes early... in the next class, we'll be doing a lot of quizzes on this.' No specific practice tasks or puzzles were assigned for the students to complete before ending."
—,Rapport Building,No,"The tutor jumps straight into the lesson topic ('today, we'll be learning about King Apojis') after a brief wait for students to join, without asking about school, weekend, or hobbies."
3,Introduction Quality,Yes,"Tutor introduces the topic: 'So, today, we'll be learning about King Apojis. Now, have you heard about it?' and proceeds to explain the concept using a board and video."
4,Setup & Puzzles Engagement,Yes,"Tutor sets up positions on the board ('Can you see the board?') and asks students to solve them ('Please tell me how you will take the king off', 'Who will win this?')."
5,General Interactive Flow,Yes,"The tutor engages in a back-and-forth dialogue, asking questions, waiting for answers, and correcting students (e.g., 'No, just take the opposition.' 'Why black will move the king to e7?')."
6,Session Conclusion,No,"The conclusion is abrupt and disorganized. The tutor mentions ending early, promises quizzes for the next class, and ends with 'Bye-bye. Sneeze.' without a clear summary or explicit homework assignment."
7,Joiner Introductions,Yes,"Tutor addresses a late joiner: 'And, you joined late, so do you have any idea about Hokojisin?' and 'Do you have, do you know what is the opposition?'"


In [26]:
sample_vtt = r"class recordings_\average_\B516 Beginner-transcript-1.1.vtt"

if os.path.exists(sample_vtt):
    extracted, score, rule = audit_and_display(sample_vtt)
else:
    print(f"File not found: {sample_vtt}")

#,Feature,State,Evidence Quote
1,Questions Asked Individually,Yes,"Tutor asks specific students by name: 'Krieshea, develop your pieces.', 'Sharon? Did you find…', 'Joanna? You're not playing?'"
2,Homework Assignment,Yes,Tutor states: 'Don't forget to do the 4 per kerki.' (likely referring to 4 puzzles/exercises) before ending the class.
—,Rapport Building,No,"The tutor jumps straight into reviewing the previous topic ('Which topic we were discussed in the last class?') without asking about school, weekends, or personal well-being."
3,Introduction Quality,Yes,Tutor welcomes students and reviews the previous lesson's topic (Special moves/En passant) to set the context for the current session.
4,Setup & Puzzles Engagement,Yes,Tutor discusses specific puzzle solutions ('Did you find the move?') and guides students through en passant scenarios on the board.
5,General Interactive Flow,Yes,"Students actively participate with answers like 'Knight to E4?', 'Move the pawn in e5 to D6', and the tutor responds to their inputs throughout the class."
6,Session Conclusion,No,"The class ends abruptly due to time ('the time is up'), with a brief goodbye and homework reminder, lacking a structured summary or wrap-up of key concepts."
7,Joiner Introductions,Yes,"Tutor welcomes a new student: 'So, Sharon, hi! ... So, welcome to today's session.' and asks if they were absent previously."


# WEAk

In [28]:
sample_vtt = r"class recordings_\weak\B558 Beginner -transcript-1.1.vtt"

if os.path.exists(sample_vtt):
    extracted, score, rule = audit_and_display(sample_vtt)
else:
    print(f"File not found: {sample_vtt}")

#,Feature,State,Evidence Quote
1,Questions Asked Individually,Yes,"Tutor calls 'Edlin' and 'Navika' by name multiple times to provide moves, e.g., 'Okay, tell me your move, Edley.' and 'Navika? ... Tell me your move.'"
2,Homework Assignment,No,The class ends with 'Next class we will be space. Okay?' and no specific puzzles or practice tasks are assigned.
—,Rapport Building,No,"The tutor asks 'How are you?' but immediately transitions to 'Okay, Eddie, today we learned... book.' without engaging in social check-ins about school or weekends."
3,Introduction Quality,Yes,"Tutor clearly states the lesson objective: 'Today, we learn how to play with the king, pawn, and queen, okay?'"
4,Setup & Puzzles Engagement,Yes,"Tutor sets up positions for students to solve, e.g., 'In the game, we just collect the stars... tell me your move' and 'white to play. Eddling.'"
5,General Interactive Flow,Yes,"The tutor actively solicits moves from students one by one and provides immediate feedback, e.g., 'Okay, excellent move, well played.'"
6,Session Conclusion,No,"The conclusion is disorganized, focusing on technical issues ('Navika, please disconnect the one device') and ending abruptly without a proper summary or goodbye to all students."
7,Joiner Introductions,—,No new students joined during the transcript; all participants were present at the start.


In [27]:
sample_vtt = r"class recordings_\weak\B482 Intermediate -transcript-1.1.vtt"

if os.path.exists(sample_vtt):
    extracted, score, rule = audit_and_display(sample_vtt)
else:
    print(f"File not found: {sample_vtt}")

#,Feature,State,Evidence Quote
1,Questions Asked Individually,Yes,"Tutor calls specific students by name to answer: 'Ruthrika, could you tell what do we play here', 'Naythan, what can we play here?', 'Krish, you can just go to the 19th move ones'"
2,Homework Assignment,No,"The class ends with a tournament and a brief analysis of a game for Krish, but no explicit practice tasks or puzzles are assigned for the students to complete outside of class."
—,Rapport Building,No,"The tutor jumps straight into chess content ('Alright, have you remembered what are we doing in the last class?') without any social check-in regarding school, weekend, or hobbies."
3,Introduction Quality,No,The opening is disorganized with technical issues and immediate recall of previous topics without a structured introduction or agenda for the current session.
4,Setup & Puzzles Engagement,Yes,The tutor explicitly states 'We can just do some of the puzzles' and proceeds to solve multiple tactical puzzles with the students.
5,General Interactive Flow,Yes,"The tutor actively engages multiple students (Rajiv, Ebrahim, Naythan, Krish) in solving puzzles and analyzing a game, asking for their moves and feedback."
6,Session Conclusion,No,"The conclusion is abrupt and disorganized, ending with 'I'm just leaving the session' and 'bye' without a proper summary or structured closing."
7,Joiner Introductions,Yes,Tutor asks new student Krish: 'Krish are you new to the class? ... What's your good name? ... How old are you?'


# Quality issue

In [29]:
sample_vtt = r"class recordings_\quality issue_\B392 Beginner-transcript-1.1.vtt"

if os.path.exists(sample_vtt):
    extracted, score, rule = audit_and_display(sample_vtt)
else:
    print(f"File not found: {sample_vtt}")

#,Feature,State,Evidence Quote
1,Questions Asked Individually,Yes,"Tutor explicitly assigns puzzles to specific students by name: 'this one goes to Avishek', 'This one goes to daksh', 'This one goes to Aadinath', 'This one goes to Aidan'."
2,Homework Assignment,No,"The class ends with individual one-on-one discussions about tournament results and ratings for Aadinath and Avishek. No explicit practice tasks, puzzles, or homework are assigned to the group or individuals before the session concludes."
—,Rapport Building,Yes,"Tutor engages in social check-ins regarding recent tournaments: 'How was the competition? Was it very tough?', 'Which position did you get?', and offers life advice: 'eat good food, easily digestible food... manage your energies'."
3,Introduction Quality,Yes,"Tutor reviews the previous topic: 'Okay, we were doing Sicilian defense, right?' and asks students to recall versions: 'Do you remember all the version of Sicilian defense?'"
4,Setup & Puzzles Engagement,Yes,"Tutor sets up a puzzle session: 'So we'll go to puzzle theme, and… can we do Checkmate in 2? Can you guys try?' and proceeds to solve multiple puzzles with students."
5,General Interactive Flow,Yes,"The class involves continuous back-and-forth where students attempt moves ('Queen to A1', 'Knight to G5') and the tutor provides immediate feedback ('Very good', 'Well done', 'Good')."
6,Session Conclusion,No,"The conclusion is fragmented and lacks a structured summary of the lesson's key takeaways (Desperado/Deflection). It devolves into individual administrative/social chats with specific students (Aadinath, Avishek) and ends abruptly without a formal wrap-up for the whole class."
7,Joiner Introductions,Yes,"Tutor acknowledges a new joiner: 'Hi, Avishek.' and later 'Aadinath also joined. I think everybody joined, right?'"


In [30]:
sample_vtt = r"class recordings_\quality issue_\B441 Beginner-transcript-1.1.vtt"

if os.path.exists(sample_vtt):
    extracted, score, rule = audit_and_display(sample_vtt)
else:
    print(f"File not found: {sample_vtt}")

#,Feature,State,Evidence Quote
1,Questions Asked Individually,Yes,"Yasar calls multiple students by name to check knowledge: 'Jaish, you know what is opposition?', 'Yajnas? You know what is opposition?', 'AARAV, you know, opposition?', 'Akshat, you know opposition?', 'David, you know opposition?', 'Mihira, you know opposition?'"
2,Homework Assignment,No,"The class ends with a live tournament/competition ('So everyone play the tournament till that, okay, class, and in 20 minutes'). No explicit practice puzzles or homework tasks are assigned for outside the class time."
—,Rapport Building,No,"The tutor jumps straight into technical chess topics (Zugzwang, Opposition) after a brief technical check-in. There is no social check-in regarding school, weekend, or hobbies."
3,Introduction Quality,No,"The opening is chaotic with technical audio issues and name confusion. The tutor does not provide a clear, structured introduction to the lesson's objectives before starting the content."
4,Setup & Puzzles Engagement,Yes,"The tutor sets up specific positions for analysis: 'So, look at this. How important is a posture? Wait, look at this.' and 'You are playing as a white... It's just an exam puzzle.'"
5,General Interactive Flow,Yes,"The tutor actively engages students with questions and calls on them by name to answer or play: 'Yajnas? You try, you have a control, try to win this game.' and 'Jaish, you losing a material or not, Jaish?'"
6,Session Conclusion,No,"The conclusion is disorganized, focusing on managing the live tournament logistics ('Please join quicker', 'Otherwise, you are not paired!') rather than summarizing the lesson or providing a clear closing statement."
7,Joiner Introductions,No,New joiners like David and Jesh are acknowledged for technical reasons (audio/name correction) but are not invited to introduce themselves socially. The tutor focuses on getting them into the flow of the class/tournament.
